# Phase 5 v7d Multi-Axis Finite Quadratic Module Test
No file IO. Each cell defines and tests one claim.

In [ ]:
import itertools, math, numpy as np
def elems(mods): return list(itertools.product(*[range(d) for d in mods]))
def lcm(a,b): return abs(a*b)//math.gcd(a,b)
def Bval(x,y,mods,cross):
    val=sum(x[i]*y[i]/mods[i] for i in range(len(mods)))
    for (i,j),c in (cross or {}).items():
        L=lcm(mods[i],mods[j]); val += c*(x[i]*y[j]+x[j]*y[i])/L
    return val
def test(mods,cross=None):
    E=elems(mods); n=len(E); K=np.array([[np.exp(-2j*np.pi*Bval(x,y,mods,cross))/math.sqrt(n) for y in E] for x in E])
    I=np.eye(n); idx={x:i for i,x in enumerate(E)}; R=np.zeros((n,n),complex)
    for i,x in enumerate(E): R[i,idx[tuple((-x[j])%mods[j] for j in range(len(mods)))]]=1
    G=np.array([np.exp(1j*np.pi*Bval(x,x,mods,cross)) for x in E])
    pol=0
    for x in E:
      for y in E:
        z=tuple((x[j]+y[j])%mods[j] for j in range(len(mods)))
        pol=max(pol, abs(G[idx[z]]/(G[idx[x]]*G[idx[y]])-np.exp(2j*np.pi*Bval(x,y,mods,cross))))
    return max(np.max(np.abs(K@K.conj().T-I)), np.max(np.abs(K@K-R)), pol)


In [ ]:
res=test([8,8],{(0,1):2})
print('CLAIM cross-axis 8x8 c=2 PASS' if res < 1e-9 else 'FAIL', res)


In [ ]:
res=test([8,12],{(0,1):6})
print('CLAIM mixed-axis 8x12 c=6 PASS' if res < 1e-9 else 'FAIL', res)


In [ ]:
res=test([12,12],{(0,1):5})
print('NEGATIVE singular coupling FAILS AS EXPECTED' if res > 1e-3 else 'UNEXPECTED PASS', res)
